# NeuroMind Phase 1 — Binary speech and fusion
Train and evaluate only Non-Dementia and Dementia with matched patient records. A missing modality or a synthetic label stops fusion training. These cells have not been executed in this ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/NeuroMind/12_Web_Application/NeuroMind_fullstack')
assert (PROJECT_ROOT / 'training').is_dir(), f'Update PROJECT_ROOT: {PROJECT_ROOT}'
%cd $PROJECT_ROOT
!pip -q install -r training/requirements-phase1.txt

In [ ]:
NEUROMIND = Path('/content/drive/MyDrive/NeuroMind')
AUDIO_ROOT = next((p for p in NEUROMIND.rglob('Speech Dataset*') if p.is_dir()), None)
MEMORY_CSV = NEUROMIND / '01_Datasets/Memory Dataset/Memory_Dataset.csv'
ATTENTION_CSV = NEUROMIND / '01_Datasets/Attention Dataset/Attention_Dataset.csv'
LIFESTYLE_CSV = NEUROMIND / '01_Datasets/Lifestyle Dataset/Lifestyle_Dataset.csv'
SPEECH_LABEL_MAP = PROJECT_ROOT / 'training/speech_label_map.example.json'
FUSION_LABEL_MAP = PROJECT_ROOT / 'training/fusion_label_map.example.json'
SEQUENCE_DIR = NEUROMIND / '03_Features/final_speech_sequences'
SPEECH_OUTPUT = NEUROMIND / '04_Models/Speech_Final'
FUSION_DATASET = NEUROMIND / '03_Features/final_fusion_dataset.csv'
FUSION_OUTPUT = NEUROMIND / '04_Models/Multimodal_Final'
for name, path in [('audio', AUDIO_ROOT), ('memory', MEMORY_CSV), ('attention', ATTENTION_CSV), ('lifestyle', LIFESTYLE_CSV)]:
    print(name, path, bool(path and path.exists()))
assert AUDIO_ROOT and AUDIO_ROOT.exists(), 'Point AUDIO_ROOT to the real speech folder.'
assert all(p.exists() for p in [MEMORY_CSV, ATTENTION_CSV, LIFESTYLE_CSV]), 'Provide real patient-aligned tabular files; existing synthetic demo tables do not qualify.'


In [ ]:
!python -m training.prepare_speech_embeddings --audio-root "$AUDIO_ROOT" --output-dir "$SEQUENCE_DIR" --label-map "$SPEECH_LABEL_MAP" --resume

In [ ]:
# Declare research release gates before looking at held-out test patients.
# These example cutoffs do not establish clinical validity.
!python -m training.train_speech --manifest "$SEQUENCE_DIR/speech_sequence_manifest.csv" --output-dir "$SPEECH_OUTPUT" --train-frame-mask 0.03 --min-test-macro-f1 0.70 --min-dementia-recall 0.80


In [ ]:
!python -m training.build_fusion_dataset --speech "$SPEECH_OUTPUT/speech_patient_embeddings.csv" --memory "$MEMORY_CSV" --attention "$ATTENTION_CSV" --lifestyle "$LIFESTYLE_CSV" --output "$FUSION_DATASET" --label-map "$FUSION_LABEL_MAP"

In [ ]:
!python -m training.train_fusion --dataset "$FUSION_DATASET" --speech-artifact "$SPEECH_OUTPUT/neuromind_speech_bilstm_attention.pt" --output-dir "$FUSION_OUTPUT"

In [ ]:
SPEECH_ARTIFACT = SPEECH_OUTPUT / 'neuromind_speech_bilstm_attention.pt'
FUSION_ARTIFACT = FUSION_OUTPUT / 'neuromind_multimodal_transformer_binary.pt'
!python -m training.validate_phase1_artifacts --speech-artifact "$SPEECH_ARTIFACT" --fusion-artifact "$FUSION_ARTIFACT"